# Change Data Feed (CDF) med Delta Sharing

Dette notebooken demonstrerer hvordan du bruker **Change Data Feed (CDF)** til å spore endringer i Delta-tabeller over tid.

In [1]:
! pip install -r requirements.txt

In [2]:
import os
import json
import delta_sharing
import delta_sharing
import pandas as pd
from google.cloud import storage
from datetime import datetime, timedelta

from src.auth import generate_access_token
from src.utils import (
    fetch_config_share,
    create_credentials_config,
)

In [3]:
project_id = "innsikt-data-dev-ec28"
project_num = "614733074632"
provider_full_identifier = f"projects/{project_num}/locations/global/workloadIdentityPools/skyporten-bi-dev/providers/skyporten-bi-provider-dev"
random_id = "9ivj"
schema_name = "matrikkel_silver_v1_ext"
share_name = f"{random_id}-dev"

CREDENTIALS_PATH = "credentials.json"
TOKEN_PATH = "tmp_maskinporten_token.txt"
CONFIG_PATH = "configs/config.json"
SOURCE_PATH = "share/config.share" #fil

In [4]:
create_credentials_config(provider_full_identifier, TOKEN_PATH, CREDENTIALS_PATH)

Created credential configuration file [credentials.json].


In [5]:
with open(CONFIG_PATH, 'r') as file:
    config = json.load(file)

token = generate_access_token(
    kid=config.get('kid'),
    scope=config.get('scope'),
    certname=config.get('certname'),
    audience=config.get('audience'),
    client_id=config.get('client_id'),
    token_url=config.get('url'),
)

s = token.get("access_token", "")

print(f"Generated token: {s}")

with open(TOKEN_PATH, 'w') as file:
    file.write(s)

Generated token: eyJraWQiOiJiZFhMRVduRGpMSGpwRThPZnl5TUp4UlJLbVo3MUxCOHUxeUREbVBpdVQwIiwiYWxnIjoiUlMyNTYifQ.eyJhdWQiOiJodHRwczovL3NreXBvcnRlbi5rYXJ0dmVyay5ubyIsInN1YiI6IjAxOTI6OTcxMDQwMjM4O2thcnR2ZXJrOm1hdHJpa2tlbC5iZXJldHRpZ2V0aW50ZXJlc3NlIiwic2NvcGUiOiJrYXJ0dmVyazptYXRyaWtrZWwuYmVyZXR0aWdldGludGVyZXNzZSIsImlzcyI6Imh0dHBzOi8vdGVzdC5za3kubWFza2lucG9ydGVuLm5vIiwiY2xpZW50X2FtciI6InByaXZhdGVfa2V5X2p3dCIsInRva2VuX3R5cGUiOiJCZWFyZXIiLCJleHAiOjE3NTk5Mjc3MjUsImlhdCI6MTc1OTkyNzY5NSwiY2xpZW50X2lkIjoiMWIxOWY0N2YtODBmNC00ZTM0LWE3ODMtODQ4ZWM0YjI5YTU2IiwianRpIjoiUlR1Wm9FU0NhbFk3dGsyWmxndDNmZldMbThHUVBXSlBSTk4wWlFycVBtNCIsImNvbnN1bWVyIjp7ImF1dGhvcml0eSI6ImlzbzY1MjMtYWN0b3JpZC11cGlzIiwiSUQiOiIwMTkyOjk3MTA0MDIzOCJ9fQ.MeztyfkL1esAFUbACUeO0zsQ2FM1pvb1FaX26BFBo9ja18c01sIYAb2sOzCq0oI4542pouaivrz8G4GpOox4vwsoN-O1e-EqxxnRSbWfGZydRj5x3f5zD0UQxvdBFpMI1F8xtujEDY9_4lNMeS7JDYdYEJDXtx7rov88BFLWkQDUbxHAsIhnnA8uZQMnBWaq5A_WTozWpU7I7m9RHqSCIsmO1GYE1vPNS2pzzYFHvVpfMBGaPhZ_x4xXuqKvjY1bB_LAjR5bp8NFbk8cwf2Cck0SP3jWazXCC

In [6]:
with open(CONFIG_PATH, 'r') as file:
    config = json.load(file)

token = generate_access_token(
    kid=config.get('kid'),
    scope=config.get('scope'),
    certname=config.get('certname'),
    audience=config.get('audience'),
    client_id=config.get('client_id'),
    token_url=config.get('url'),
)

s = token.get("access_token", "")

print(f"Generated token: {s}")

with open(TOKEN_PATH, 'w') as file:
    file.write(s)

Generated token: eyJraWQiOiJiZFhMRVduRGpMSGpwRThPZnl5TUp4UlJLbVo3MUxCOHUxeUREbVBpdVQwIiwiYWxnIjoiUlMyNTYifQ.eyJhdWQiOiJodHRwczovL3NreXBvcnRlbi5rYXJ0dmVyay5ubyIsInN1YiI6IjAxOTI6OTcxMDQwMjM4O2thcnR2ZXJrOm1hdHJpa2tlbC5iZXJldHRpZ2V0aW50ZXJlc3NlIiwic2NvcGUiOiJrYXJ0dmVyazptYXRyaWtrZWwuYmVyZXR0aWdldGludGVyZXNzZSIsImlzcyI6Imh0dHBzOi8vdGVzdC5za3kubWFza2lucG9ydGVuLm5vIiwiY2xpZW50X2FtciI6InByaXZhdGVfa2V5X2p3dCIsInRva2VuX3R5cGUiOiJCZWFyZXIiLCJleHAiOjE3NTk5Mjc3MjUsImlhdCI6MTc1OTkyNzY5NSwiY2xpZW50X2lkIjoiMWIxOWY0N2YtODBmNC00ZTM0LWE3ODMtODQ4ZWM0YjI5YTU2IiwianRpIjoiSnE2RG9NdXVaa3A3Rnl1LVRFUU8wOVJzcWhCWDZpUFpkUlBSZ3RReDhndyIsImNvbnN1bWVyIjp7ImF1dGhvcml0eSI6ImlzbzY1MjMtYWN0b3JpZC11cGlzIiwiSUQiOiIwMTkyOjk3MTA0MDIzOCJ9fQ.pIPgk4K4yrBFnQuNXBl_arqjRESnR2aiwrB-Afg9ZfntidCHJFlnNPEeRF1gje7ixxuamBpnMON6PISn4AQDmLCMnXl1LHaCBdqCMmBqqHc555M2jKnKbx8dZgODnAMErCL6BRnPbLLRcySg2se8XcUfA0TL7e3jv8tCXjC-nGMRGACMHQtGh9YI9Y2B6g0LTAlEhEcEPW3EbRQVx4xFlL08m_x88HzqmYOhL-Uf28V0Je8Dl98m3isPHh1zD-KVLMEVaki4XSAAKO1JcMUnbPwJJhBwmb30g

In [7]:
os.makedirs("share", exist_ok=True)
bucket_id = f"sp-{project_id}-{random_id}"
fetch_config_share(project_id, bucket_id, SOURCE_PATH, CREDENTIALS_PATH)

Valid Delta Sharing configuration found.
config.share er gyldig


In [8]:
# Spesifiser sti til din Delta Sharing config-fil
# Denne filen inneholder credentials og endpoint for din share
CONFIG_FILE = "share/config.share"  # Endre til din config-fil

In [9]:
# Koble til Delta Sharing
sharing_client = delta_sharing.SharingClient(CONFIG_FILE)
tables = sharing_client.list_all_tables()

if not tables:
    raise Exception("❌ Ingen tabeller funnet i sharen")

# Filtrer bort tabeller som ikke er gode for demonstrasjon
# (kode-tabeller, nøkkel-tabeller, krypterte tabeller)
not_valid_table_names = ["kode", "keys", "encrypted"]
valid_examples_tables = [
    table for table in tables
    if not any(substr in table.name for substr in not_valid_table_names)
]

# Velg første egnede tabell (eller første tabell hvis ingen egnede finnes)
table = valid_examples_tables[0] if len(valid_examples_tables) > 0 else tables[0]

# Bygg full tabell-URL for Delta Sharing
table_url = f"{CONFIG_FILE}#{table.share}.{table.schema}.{table.name}"

print(f"✓ Valgt tabell: {table.name}")
print(f"  Share: {table.share}")
print(f"  Schema: {table.schema}")
print(f"  Full URL: {table_url}")

✓ Valgt tabell: dim_kulturminner
  Share: 9ivj-dev
  Schema: matrikkel_silver_v1_ext
  Full URL: share/config.share#9ivj-dev.matrikkel_silver_v1_ext.dim_kulturminner


##  Opprett Spark Session



In [10]:
from pyspark.sql import SparkSession

def ensure_spark():
    global spark
    # Gjenbruk hvis 'spark' finnes og lever
    if 'spark' in globals():
        try:
            _ = spark.version
            print("✅ Gjenbruker eksisterende SparkSession")
            return spark
        except Exception:
            pass  # faller gjennom og oppretter på nytt

    print("🚀 Oppretter ny SparkSession med Delta Sharing")
    spark = (
        SparkSession.builder
        .appName("DeltaSharingCDF")
        # .master("local[*]")  # bruk bare lokalt; ikke i Databricks
        .config("spark.jars.packages", "io.delta:delta-sharing-spark_2.12:3.1.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
    )
    return spark

spark = ensure_spark()


🚀 Oppretter ny SparkSession med Delta Sharing
:: loading settings :: url = jar:file:/Users/simen/prosjekter/kartverket/kv-deltashare-examples/venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/simen/.ivy2/cache
The jars for the packages stored in: /Users/simen/.ivy2/jars
io.delta#delta-sharing-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c0febaa9-dd5f-4df7-b3de-9c72ae7314fc;1.0
	confs: [default]
	found io.delta#delta-sharing-spark_2.12;3.1.0 in central
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found io.delta#delta-sharing-client_2.12;1.0.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.2 in central
	found commons-codec#commons-codec;1.11 in central
:: resolution report :: resolve 123ms :: artifacts dl 27ms
	:: modules in use:
	commons-codec#commons-codec;1.11 from central in [default]
	commons-logging#commons-logging;1.2 from central in [default]
	io.delta#delta

# Hent endringer via CDF

1. Beregner et starttidspunkt basert på antall timer tilbake i tid.  
2. Leser endringer fra tabellen siden dette tidspunktet.  
3. Hvis det finnes endringer, grupperes de etter endringstype (`insert`, `update`, `delete`).  
4. Viser en oppsummering av antall endringer per type.  
5. Viser inntil 10 eksempler for hver endringstype.


In [ ]:
from pyspark.sql import SparkSession, functions as F
lookback_hours = 24
start_ts = (datetime.utcnow() - timedelta(hours=lookback_hours)).strftime("%Y-%m-%dT%H:%M:%S.%fZ")

cdf = (
    spark.read.format("deltaSharing")
    .option("responseFormat", "delta")
    .option("readChangeFeed", "true")
    .option("startingTimestamp", start_ts)
    .load(table_url)
)

if not cdf.rdd.isEmpty():
    change_summary = cdf.groupBy("_change_type").count().orderBy("_change_type")
    change_summary.show(truncate=False)

    for row in change_summary.collect():
        change_type = row["_change_type"]
        cdf.filter(F.col("_change_type") == change_type).show(10, truncate=False)


/var/folders/sh/t1kb_fwn67l_f8zrzb26bct40000gn/T/ipykernel_58915/3861382531.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_ts = (datetime.utcnow() - timedelta(hours=lookback_hours)).strftime("%Y-%m-%dT%H:%M:%S.%fZ")


+----------------+-----+
|_change_type    |count|
+----------------+-----+
|insert          |432  |
|update_postimage|415  |
|update_preimage |415  |
+----------------+-----+



NameError: name 'F' is not defined